# 0. Tools & Tool Calling Overview

Before the individual pieces, let's understand **what a tool is**, **why tool calling exists**, and
the **full loop** from user question to final answer. Learn this once and every later file clicks.

---

## 1. Simple Definition

> **Kid version:** Imagine a super-smart friend who's been locked in a room since 2023. They know a
> lot, but they can't look out the window, use a calculator, or check your fridge. So you give them a
> **walkie-talkie** connected to helpers outside. When they need something they can't do, they radio:
> "Please check the weather!" A helper does it and radios back the answer.
>
> - The **helpers** (weather-checker, calculator, fridge-looker) = **tools**.
> - The **walkie-talkie routine** of asking and getting answers = **tool calling**.

**Professional definition:** A *tool* is a function, paired with a name and a description, that an LLM
can choose to invoke. *Tool calling* is the protocol where the model outputs a structured request to
run a tool with specific arguments; your application executes it and returns the result to the model.

```python
from langchain_core.tools import tool

@tool
def add(a: int, b: int) -> int:
    """Add two numbers."""
    return a + b

# `add` is now a Tool the model can be given and can choose to call.
```

---

## 2. Why Does Tool Calling Exist?

**The problem:** An LLM has three big limitations:
1. **Frozen knowledge** — it doesn't know today's news, prices, or your private data.
2. **No actions** — it can't send an email, run code, or write to a database.
3. **Unreliable at exact tasks** — math, counting, precise lookups (it *predicts* text, it doesn't
   *compute*).

Tools fix all three by letting the model **delegate** to real code.

### Before tool calling (the model guesses)

```python
model.invoke("What is 24387 * 9823?")
# → "Approximately 239 million" — often WRONG; the model isn't a calculator.

model.invoke("What's the weather in Paris right now?")
# → "I can't access real-time data." — it simply can't.
```

### After tool calling (the model delegates)

```python
# Give it a calculator tool and a weather tool.
# For math → it calls multiply(24387, 9823) → gets the EXACT answer.
# For weather → it calls get_weather('Paris') → gets LIVE data.
```

The model becomes a **router/orchestrator**: it decides *which* capability is needed and *with what
arguments*, while real code does the precise work.

**Where you'll use it:** agents, RAG that can search, database Q&A, calculators, API integrations,
booking/ordering assistants, code execution — anything where the LLM must *do* rather than only *say*.

---

## 3. Real-Life Analogy

A **manager delegating to specialists** 👔. A good manager doesn't personally do every task — they
know *who* to ask. "Accounting, run these numbers." "IT, restart the server." The manager (LLM) reads
the situation and delegates to the right specialist (tool), then combines the results into a decision.

Other analogies: a **doctor ordering lab tests** (they interpret results but don't run the machines),
a **chef calling out to line cooks**, a **detective sending evidence to the forensics lab**.

---

## 4. Where Tools Fit in LangChain Architecture

```
                        BaseTool                     ← the base class for every tool
                            │
        ┌───────────────────┼─────────────────────┐
        ▼                   ▼                     ▼
   @tool decorator     StructuredTool          Your subclass of BaseTool
        │                   │                     │
        └───────────────────┴─────────────────────┘
                            │  all produce a Tool with: name, description, args_schema
                            ▼
              model.bind_tools([...])                ← attach tools to a chat model
                            │
                            ▼
              THE TOOL-CALLING LOOP    
              tool_calls → execute → ToolMessage → final answer
```

Three ways to make a tool, one way to describe it to the model, one way to attach it, and one loop to run it.

---

## 5. Internal Working — the full loop

This is the single most important diagram in the notebook:

```
  ① USER asks something
     "What's 3 times 12, and the weather in Paris?"
              │
              ▼
  ② MODEL (with tools bound) decides it needs tools and returns an AIMessage with tool_calls:
     AIMessage(content="", tool_calls=[
        {name: "multiply",    args: {a: 3, b: 12}, id: "call_1"},
        {name: "get_weather", args: {city: "Paris"}, id: "call_2"},
     ])
     ⚠️ The model does NOT run anything — it only REQUESTS.
              │
              ▼
  ③ YOUR CODE executes each requested tool:
     multiply(3, 12)      → 36
     get_weather("Paris") → "18°C, sunny"
              │
              ▼
  ④ Wrap each result in a ToolMessage (tagged with the matching tool_call_id):
     ToolMessage(content="36",            tool_call_id="call_1")
     ToolMessage(content="18°C, sunny",   tool_call_id="call_2")
              │
              ▼
  ⑤ SEND the conversation (user + AIMessage + ToolMessages) back to the model
              │
              ▼
  ⑥ MODEL now has the results and writes the final answer:
     "3 × 12 is 36, and it's 18°C and sunny in Paris."
```

Key insight: it's a **two-round-trip** conversation. Round 1 the model *asks* for tools; round 2 it
*answers* using the results. An **agent** just automates this loop, possibly for many rounds.

---

## 6. The Core Vocabulary

### `Tool`

**Definition:** A callable + `name` + `description` + `args_schema` that the model can invoke.

**Why it exists:** It's the unit of capability you give the model.

**Real-life use case:** One specialist the manager can delegate to.

```python
@tool
def get_weather(city: str) -> str:
    """Get the current weather for a city."""
    return f"18°C and sunny in {city}"
```

---


### `bind_tools()`

**Definition:** Attaches a list of tools to a chat model so it *can* call them.

**Why it exists:** The model must be told which tools exist (and their schemas) before it can request
them.

```python
model_with_tools = model.bind_tools([get_weather, add])
```

---

### `tool_call`

**Definition:** The model's structured **request** to run a tool: `{name, args, id}`. Found on
`AIMessage.tool_calls`.

**Why it exists:** It's how the model communicates "run this, with these arguments."

**Real-life use case:** The written work-order the manager hands a specialist.

```python
ai_msg = model_with_tools.invoke("weather in Paris?")
ai_msg.tool_calls   # [{'name': 'get_weather', 'args': {'city': 'Paris'}, 'id': 'call_abc'}]
```

---

### `ToolMessage`

**Definition:** The message carrying a tool's **result** back to the model, tagged with
`tool_call_id`.

**Why it exists:** So the model can match each result to the request it made.

**Real-life use case:** The specialist's report sent back to the manager.

```python
from langchain_core.messages import ToolMessage
ToolMessage(content="18°C and sunny", tool_call_id="call_abc")
```

---

## 7. A crucial safety point

**The model never executes your code.** It only *emits a request* (`tool_calls`). **Your program**
decides whether and how to run it. This means:
- You can validate/authorize before running (e.g. don't let it delete files).
- You can sandbox dangerous tools (code execution, shell).
- You stay in control — the LLM is an advisor, not an admin.